In [4]:
from transformers import AutoTokenizer,AutoModelForCausalLM,TrainingArguments,Trainer
from peft import LoraConfig,get_peft_model,TaskType
from datasets import load_dataset

In [2]:
base_model="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [6]:
tokenizer=AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

In [9]:
model_path="./lora_trained_model/checkpoint-1"

In [10]:
instructed_model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


DEBUG BitBLAS import failed: No module named 'bitblas'                         


DEBUG Skipping qlinear module import `bitblas_target_detector`: No module named 'thefuzz'


INFO  

_____/\\\\\\\\\\\\__/\\\\\\\\\\\\\____/\\\\\\\\\\\\\\\______________________/\\\________/\\\\____________/\\\\_______________________/\\\__________________/\\\\\\____
 ___/\\\//////////__\/\\\/////////\\\_\///////\\\/////____________________/\\\\/\\\\____\/\\\\\\________/\\\\\\______________________\/\\\_________________\////\\\____
  __/\\\_____________\/\\\_______\/\\\_______\/\\\_______________________/\\\//\////\\\__\/\\\//\\\____/\\\//\\\______________________\/\\\____________________\/\\\____
   _\/\\\____/\\\\\\\_\/\\\\\\\\\\\\\/________\/\\\________/\\\\\\\\\\\__/\\\______\//\\\_\/\\\\///\\\/\\\/_\/\\\_____/\\\\\___________\/\\\______/\\\\\\\\_____\/\\\____
    _\/\\\___\/////\\\_\/\\\/////////__________\/\\\_______\///////////__\//\\\______/\\\__\/\\\__\///\\\/___\/\\\___/\\\///\\\____/\\\\\\\\\____/\\\/////\\\____\/\\\____
     _\/\\\_______\/\\\_\/\\\___________________\/\\\______________________\///\\\\/\\\\/___\/\\\____\///_____\/\\\__/\\\__\//\\\__/\\\////\\\___/\

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [20]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."
     

In [21]:
inputs=tokenizer(prompt,return_tensors="pt")

In [22]:
outputs=instructed_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [23]:
print("\n Instructed Model Output:\n")
print(tokenizer.decode(outputs[0],skip_special_tokens=True))


 Instructed Model Output:

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry. 2017-08-31 23:45:23
3649 AI Drug Discovery and Development - AI Drug Discovery and Development - AI Drug Discovery and Development Introduction to Artificial Intelligence (AI) The process of drug discovery is long, complex, and highly technical, so it's no wonder that many people struggle with the task. In fact, a recent study found that only one in


In [24]:
import trl

In [25]:
from trl import DPOTrainer
from peft import PeftModel
import torch

In [39]:
base_model="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

instruction_checkpoint="/lora_trained_model/checkpoint-1"

In [27]:
dataset=load_dataset("csv",data_files="./pharma_preference_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [30]:
#get_peft_model-creates new peft model

#PeftModel-loads already-trained lora model for training

lora_config=LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","v_proj"],
    bias="none")

In [31]:
pref_peft_model=get_peft_model(instructed_model,lora_config)

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [35]:
from transformers import BitsAndBytesConfig

bnb_config=BitsAndBytesConfig(
    load_in_8bit=True
)

In [37]:
model=AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/generation_config.json "HTTP/1.1 200 OK"


In [42]:
from peft import PeftModel

instruction_checkpoint = "lora_trained_model/checkpoint-1"

model = PeftModel.from_pretrained(
    model,
    instruction_checkpoint
)

In [43]:
model=model.merge_and_unload()

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\lora\bnb.py:110: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [44]:
pref_model_lora=get_peft_model(model,lora_config)

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [45]:
from trl import DPOTrainer,DPOConfig

In [47]:
import os
os.environ["WANDB-DISABLED"]="true"

In [51]:
dpo_args=DPOConfig(
    output_dir="./tiny-llama/preference-alignment",
    learning_rate=2e-5,
    gradient_accumulation_steps=8,
    logging_dir=None,
    report_to=[],
    per_device_train_batch_size=1,
    num_train_epochs=1,
    beta=0.1,
    loss_type="sigmoid",
    remove_unused_columns=False,
    bf16=False,
)

In [52]:
trainer=DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,

)

In [53]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-int

TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 47.3039, 'train_samples_per_second': 0.106, 'train_steps_per_second': 0.021, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996})

In [82]:
#Non_Instructed Model
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [83]:
model_path="lora_trained_model/checkpoint-1"
non_instructed_model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/generation_config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [84]:
inputs=tokenizer(question,return_tensors="pt")

In [85]:
non_instructed_model.config.pad_token_id = tokenizer.eos_token_id

In [86]:

outputs = non_instructed_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    do_sample=True,
    repetition_penalty=1.1,
    top_p=0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
In this article, you'll learn about Metformin's potential to fight off heart disease, diabetes, cancer and even Alzheimer's. We also examine the side effects of Metformin and how its popularity has grown over the years.
What is Metformin?
Metformin (also known as Glucophage) is a prescription drug that reduces the amount of glucose in your bloodstream by increasing insulin production in


In [87]:
#Instructed_model

model_path="instructed-model/checkpoint-1"

instructed_model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/generation_config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [88]:
inputs=tokenizer(question,return_tensors="pt")

In [89]:
outputs = instructed_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    do_sample=True,
    repetition_penalty=1.1,
    top_p=0.9
)

In [90]:
print("Model Output:\n")
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Why Metformin Should Be Taken with Food
Metformin is a generic drug that has been on the market for over 30 years, and there are many studies that demonstrate its safety and efficacy when taken at a higher dose than recommended. The drug works by reducing blood sugar levels in people with type 2 diabetes, which can help prevent complications from this disease such as kidney damage and blindness. In addition to treating type 2 diab


In [91]:

model_path="tiny-llama/preference-alignment/checkpoint-1"
pref_aligned_model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/generation_config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [92]:
inputs=tokenizer(question,return_tensors="pt")

In [93]:
outputs = pref_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    do_sample=True,
    repetition_penalty=1.1,
    top_p=0.9
)

In [94]:
print("Model Output:\n")
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Learn about some of the key differences between Type 1 and Type 2 Diabetes and understand the signs, symptoms, and risks associated with both conditions.
Find out about metabolism and how changes to your lifestyle may help you lower your risk for developing type 2 diabetes.
Discover why most people who develop diabetes have type 2 diabetes and learn how you can protect yourself from this disease.
